# AIGC 5005 · Module 3 · Worked example
## Refactoring a script into an installable package

Companion notebook to the worked example of the Module 3 learning content. Every example from that part is here, ready to run, with comments explaining what to notice.

**Estimated time:** 30 minutes

**How to use it:** read each explanation, run the cell beneath it, and compare the output with what you expected *before* you ran it. The prediction is where the learning happens. Change values and run cells again; you cannot break anything.

**Kernel:** select the kernel from your `module03` conda environment (top right in VS Code or Jupyter). To make one: `conda create -n module03 python=3.11 ipykernel`, then `conda activate module03`. If VS Code offers to install `ipykernel`, accept.

## 0. Set up a scratch folder

We build both versions of the catalogue (the flat "before" script and the packaged "after" project) inside `scratch_refactor`, then compare them. Everything is removed by the last cell.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

SCRATCH = Path("scratch_refactor").resolve()
if SCRATCH.exists():
    shutil.rmtree(SCRATCH)
SCRATCH.mkdir()


def run(args, cwd, env_extra=None):
    """Run a command in another Python process and return (exit code, stdout, last line of stderr)."""
    env = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}   # start with no hidden import path
    env.update(env_extra or {})
    r = subprocess.run(args, cwd=cwd, capture_output=True, text=True, env=env)
    last_err = r.stderr.strip().splitlines()[-1] if r.stderr.strip() else ""
    return r.returncode, r.stdout.strip(), last_err

## 1. The "before": one flat script

Everything is mixed into one file, the data is an untyped dictionary, and nothing records what the script needs.

In [ ]:
before = SCRATCH / "before"
before.mkdir()
(before / "catalogue_script.py").write_text('''books = []


def add_book(title, author):
    books.append({"title": title.strip().title(), "author": author, "copies": 1})


def checkout(title):
    for b in books:
        if b["title"] == title and b["copies"] > 0:
            b["copies"] -= 1
            return
    raise ValueError(f"'{title}' is not available.")


add_book("fluent python", "Luciano Ramalho")
checkout("Fluent Python")
print(books)
''')

code_, out, err = run([sys.executable, "catalogue_script.py"], cwd=before)
print(out)

### What goes wrong with a dictionary

Nothing says which keys a book must have. A misspelt or missing key is accepted without complaint and only fails later, somewhere else, when other code reads the key.

**Predict first:** which of these lines raises an error immediately?

In [ ]:
book = {"titel": "Clean Code", "author": "Robert Martin", "copies": 1}   # typo: "titel" -- accepted silently
print(book)                          # nothing complained

try:
    print(book["title"])             # the failure only appears here, later, when the key is read
except KeyError as error:            # try/except only keeps the notebook running here
    print("KeyError:", error)

## 2. The "after": a package with a class

The same behaviour, now in the layout from Part 4, with a class from Part 1, a helper module, an entry point (`main.py`) inside the package, a `pyproject.toml`, and a pinned `requirements.txt`.

In [ ]:
proj = SCRATCH / "catalogue_project"
(proj / "src/catalogue").mkdir(parents=True)
(proj / "tests").mkdir()

(proj / "src/catalogue/helpers.py").write_text('''def format_title(title):
    return title.strip().title()
''')

(proj / "src/catalogue/models.py").write_text('''from .helpers import format_title


class Book:
    """A single book in the catalogue."""

    def __init__(self, title, author, copies=1):
        self.title = format_title(title)
        self.author = author
        self.copies = copies

    def checkout(self):
        if self.copies <= 0:
            raise ValueError(f"'{self.title}' is not available.")
        self.copies -= 1
''')

(proj / "src/catalogue/__init__.py").write_text('''from .models import Book

__all__ = ["Book"]
''')

(proj / "src/catalogue/main.py").write_text('''from .models import Book


def main():
    b = Book("fluent python", "Luciano Ramalho")
    b.checkout()
    print(b.title, b.copies)   # Fluent Python 0


if __name__ == "__main__":
    main()
''')

(proj / "requirements.txt").write_text("requests==2.32.3\nipykernel==7.4.0\n")

(proj / "pyproject.toml").write_text('''[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "catalogue"
version = "0.1.0"
dependencies = [
    "requests==2.32.3",
]

[tool.setuptools.packages.find]
where = ["src"]
''')

(proj / "tests/test_models.py").write_text('''import unittest

from catalogue import Book


class TestBook(unittest.TestCase):
    def test_title_is_formatted(self):
        self.assertEqual(Book("  fluent python ", "A").title, "Fluent Python")

    def test_checkout_with_no_copies_raises(self):
        with self.assertRaises(ValueError):
            Book("x", "A", copies=0).checkout()


if __name__ == "__main__":
    unittest.main()
''')
print("project files:", sorted(str(p.relative_to(proj)) for p in proj.rglob("*") if p.is_file()))

### The class enforces the shape at creation time

Now `title` and `author` are required, so a malformed book is rejected the moment you create it, not later.

In [ ]:
sys.path.insert(0, str(proj / "src"))      # temporary: lets THIS notebook import the package (see section 3)
from catalogue import Book

try:
    Book(titel="Clean Code", author="Robert Martin")     # typo in the keyword
except TypeError as error:                  # try/except only keeps the notebook running here
    print("TypeError:", error)

good = Book("clean code", "Robert Martin", copies=1)
good.checkout()
print(good.title, good.copies)              # Clean Code 0

## 3. Why the package cannot be found yet

The package lives under `src/`, so `python -m catalogue.main` has nowhere to look for `catalogue`. This is exactly the gap `pyproject.toml` and an **editable install** close.

**Predict first:** will `python -m catalogue.main` run?

In [ ]:
code_, out, err = run([sys.executable, "-m", "catalogue.main"], cwd=proj)
print("exit code:", code_)
print(err)                                   # ... No module named 'catalogue'

### What an editable install does

`pip install -e .` reads `pyproject.toml`, finds the package under `src/`, and links it into the active environment so `import catalogue` works from anywhere. Edits to the source take effect immediately, with no reinstall.

To **see the effect without changing your environment**, the cell below tells Python where to look using `PYTHONPATH`. It is not how you would ship a project, but it shows the same result.

In [ ]:
code_, out, err = run([sys.executable, "-m", "catalogue.main"], cwd=proj, env_extra={"PYTHONPATH": str(proj / "src")})
print("exit code:", code_)
print(out)                                   # Fluent Python 0

### Do the real install (optional)

In your own project you would run this once, inside your activated conda environment (never `base`). It is switched off here so running the notebook never changes your Python environment. Set `RUN_INSTALL = True` only inside a conda environment you created for this module.

In [ ]:
RUN_INSTALL = False

if RUN_INSTALL:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(proj)], capture_output=True, text=True)
    print(r.stdout[-400:] or r.stderr[-400:])
else:
    print("Skipped. In a terminal, from the project folder:   pip install -e .")

## 4. Run the tests

Tests live in `tests/`, apart from the code they check. They import `catalogue` the same way any user would.

In [ ]:
r = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"], cwd=proj,
                   capture_output=True, text=True,
                   env={**{k: v for k, v in os.environ.items() if k != "PYTHONPATH"}, "PYTHONPATH": str(proj / "src")})
print(r.stderr.strip().splitlines()[-1])     # OK

### Try it

Take the flat script from your own Lab 02 and plan its refactor: which functions belong together in a class, which belong in a helper module, and what goes in `requirements.txt`? Sketch the folder layout in the cell below as a text tree (Lab 03 asks you to build it for real).

In [ ]:
# Sketch your planned layout here, for example as a multi-line string.

## Clean up

In [ ]:
sys.path.remove(str(proj / "src"))
for name in [n for n in sys.modules if n == "catalogue" or n.startswith("catalogue.")]:
    del sys.modules[name]
shutil.rmtree(SCRATCH)
print("Scratch folder removed:", not SCRATCH.exists())